# RBRQ operational pipeline

## Explanation and usage

Use this notebook to run the RBRQ workflow end-to-end in small, explicit steps.
You need a valid `inst_deploy_id`; if you do not know it, use `inst_deploy_id_finder.ipynb` first.

Suggested run order for operators:
1. Run Imports.
2. Run Setup.
3. Run ID lookup to validate inputs and confirm metadata for the selected deployment.
4. If you need to discover an ID first, run `inst_deploy_id_finder.ipynb` before this notebook.
5. Run only the stage cells you need (`proc_1`, `proc_2`, `imos_delivery`).

This notebook is designed for cell-by-cell execution rather than Run All.

In [ ]:
import os
import sys
from pathlib import Path
import importlib
import subprocess

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(value):
        print(value)

TOOLS_DIR = Path.cwd().resolve().parent
if not (TOOLS_DIR / "tools").exists():
    candidate = Path.cwd().resolve()
    if (candidate / "tools").exists():
        TOOLS_DIR = candidate
    elif (candidate / "mooring_proc" / "tools").exists():
        TOOLS_DIR = candidate / "mooring_proc"

if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from tools.helpers import plot_data_by_qc, plot_pressure_comparison
from tools.workflows.run_imos_delivery import run_imos_delivery
from tools.workflows.run_proc1 import run_proc1
from tools.workflows.run_proc2 import run_proc2
import tools.database_lookup as database_lookup
import pandas as pd
from tools.parsers.read_rbrq import read_rbrq


## Setup

### Working directory

Edit this path if you want the notebook to resolve relative data paths from a different root, not the repository.


In [ ]:
# Working directory
working_directory = "/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data"
os.chdir(working_directory)
print(f"Working directory: {os.getcwd()}")


### Set permissions
This step sets the umask, with options to add at different steps if required.

NOTE: umask sets the process umask to the value you pass and returns the previous umask value

In [ ]:
os.umask(0o002)

# umask 0o002 gives files 664 (-rw-rw-r--)
# umask 0o022 gives files 644 (-rw-r--r--)


### Locate instrument - files and metadata
Set the deployment identifier and instrument or use optional source override (source path+file)

In [ ]:
# Required RBRQ deployment identifier from the metadata table.
inst_deploy_id = ""

# Keep this notebook scoped to RBRQ instruments.
instrument = "RBRQ"

# Optional override for the raw input source file or directory.
proc1_source_path = None

This cell turns the options above into the shared workflow configuration used by the processing stages.


In [ ]:
def build_workflow_config():
    import tools.database_lookup as database_lookup
    database_lookup = importlib.reload(database_lookup)
    metadata_csv = str(database_lookup.DEFAULT_METADATA_CSV_PATH)
    return {
        "metadata_csv": metadata_csv,
        "metadata_source": metadata_csv,
        "inst_deploy_ID": inst_deploy_id,
        "instrument": instrument,
        "manual_qc_flags": list(globals().get("manual_qc_flags", [])),
    }


def require_ready_config(require_instrument=True):
    if require_instrument and not str(inst_deploy_id).strip():
        raise ValueError("Set inst_deploy_id to a valid RBRQ deployment identifier before running this cell.")
    return build_workflow_config()


def fmt_utc(value):
    timestamp = pd.to_datetime(value, utc=True, errors="coerce")
    if pd.isna(timestamp):
        return ""
    return timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")

### ID lookup

This cell validates setup inputs and resolves metadata for the selected `inst_deploy_id` before any processing step.

In [ ]:
# Validate setup and print metadata for the selected deployment identifier.

database_lookup = importlib.reload(database_lookup)

inst_deploy_id = str(inst_deploy_id).strip()

instrument = str(instrument).strip().upper()

selected_metadata_row = None
selected_metadata_cfg = None
selected_metadata_lines = []
lookup_found = False
lookup_error = None

try:
    _, selected_metadata_row, selected_metadata_cfg, selected_metadata_lines = database_lookup.get_instrument_context(
        None,
        inst_deploy_id,
        deployment_id=None,
    )
    lookup_found = True
    print(f"Metadata for inst_deploy_id={inst_deploy_id}:")
    for line in selected_metadata_lines:
        print(line)
except ValueError as exc:
    lookup_error = str(exc)
    print(f"ID lookup failed for inst_deploy_id={inst_deploy_id}: {lookup_error}")
    print("Use `inst_deploy_id_finder.ipynb` to search candidate IDs before rerunning this cell.")


# Proc 1

This stage produces the IMOS FV00 intermediate product. It:
1. resolves the raw source file and the trim window,
2. applies QC window logic and trims to the deployment bounds,
3. writes IMOS FV00 NetCDF (without QC flags) to proc_1 directory,
4. updates metadata CSV with the saved filename.

FV00 is not a final deliverable; it is used as input for proc_2 QC refinement.

## Proc_1 controls

Set the time window first. If you plan to review and flag data, define those settings here before running proc_1.


In [ ]:
# Optional `proc_1` time coverage overrides.
# Leave both as None to use the deployment metadata window.
proc1_time_start_override = None
proc1_time_end_override = None

## Inspect proc_1 source file

Run this cell to load the raw source file, check its start and end timestamps, and confirm the resolved input path before modifying anything.


In [ ]:
proc1_source_preview = read_rbrq(proc1_source_path, config={"metadata_row": selected_metadata_row.to_dict()})
proc1_source_path = proc1_source_preview["input_path"]
proc1_source_frame = proc1_source_preview["dataframe"]
proc1_source_start = proc1_source_frame.index.min()
proc1_source_end = proc1_source_frame.index.max()

print(f"Resolved source path: {proc1_source_path}")
print(f"Source start timestamp: {fmt_utc(proc1_source_start)}")
print(f"Source end timestamp: {fmt_utc(proc1_source_end)}")
print(f"Deployment window start: {fmt_utc(selected_metadata_row.get('time_coverage_start'))}")
print(f"Deployment window end: {fmt_utc(selected_metadata_row.get('time_coverage_end'))}")


## run_proc_1

Resolves the trim window, creates proc_1 metadata, and writes the IMOS FV00 file.
CSV is updated immediately after the output NetCDF is written successfully.


In [ ]:
workflow_config = require_ready_config()
proc1_config = dict(workflow_config)

proc1_time_start_override = globals().get("proc1_time_start_override")
proc1_time_end_override = globals().get("proc1_time_end_override")

if proc1_time_start_override not in (None, ""):
    proc1_config["time_coverage_start"] = str(proc1_time_start_override)
if proc1_time_end_override not in (None, ""):
    proc1_config["time_coverage_end"] = str(proc1_time_end_override)

resolved_start = proc1_config.get("time_coverage_start", selected_metadata_row.get("time_coverage_start"))
resolved_end = proc1_config.get("time_coverage_end", selected_metadata_row.get("time_coverage_end"))
print(f"proc_1 trim window: {fmt_utc(resolved_start)} to {fmt_utc(resolved_end)}")

proc1_result = run_proc1(proc1_config, source_path=proc1_source_path)


## proc_1 review plot

Inspect QC flags interactively before moving to proc_2.


In [ ]:
if proc1_result is None:
    raise ValueError("Run the proc_1 execution cell first.")

review_figure = plot_data_by_qc(
    proc1_result["dataset"],
    title=f"proc_1 review by QC flag: RBRQ {selected_metadata_row.get('inst_id')}",
)
review_figure.show()


## Optional pressure comparison review

Set `pressure_review_deploy_id` to the `inst_deploy_ID` of a co-deployed reference pressure instrument (e.g. a tide gauge or BPR). The cell overlays the two pressure series and displays residuals.


In [ ]:
# Optional pressure comparison review
# Set pressure_review_deploy_id to the inst_deploy_ID of a co-deployed reference instrument
# (e.g. a tide gauge or BPR) to generate a pressure-comparison plot after proc_1.
# Leave as None to skip.
pressure_review_deploy_id = None
pressure_review_file = None  # Alternative: explicit path to reference file

if proc1_result is not None and (pressure_review_deploy_id or pressure_review_file):
    pressure_fig, _, _ = plot_pressure_comparison(
        df=proc1_result["dataframe"],
        database=None,
        pressure_inst_deploy_id=pressure_review_deploy_id,
        pressure_file=pressure_review_file,
        primary_label="RBRQ",
    )
    pressure_fig.show()
else:
    print("Pressure comparison skipped (set pressure_review_deploy_id or pressure_review_file to enable).")


## Save proc_1

FV00 file has been created and CSV updated. Confirm the saved file path and ownership.


In [ ]:
proc1_output_path = Path(proc1_result["output_path"])

# optional: make the parent directory shared/group-writable
proc1_output_path.parent.chmod(0o2775) #

# set the group ownership on the file itself
subprocess.run(["chgrp", "1054842", str(proc1_output_path)], check=True)

print(f"proc_1 FV00 saved: {proc1_output_path}")


# Proc 2

This stage produces the final IMOS FV01 deliverable. It:
1. loads the proc_1 FV00 file from disk,
2. reconstructs QC variables from schema,
3. applies manual QC flags and refines the product,
4. writes IMOS FV01 NetCDF to proc_2 directory.

FV01 is the only product that should be staged for compliance validation and delivery.

In [ ]:
proc2_input = proc1_output_path
print(f"Using proc_1 FV00 input: {proc2_input}")


## Proc_2 QC controls

Set or edit manual QC windows here before running proc_2.


Example:

    {
        "qc_vars": ["variable_quality_control"],
        "flag": 4,
        "start": "yyyy-mm-ddThh:mm:ss",
        "end": "yyyy-mm-ddThh:mm:ss",
        "comment": "Spike - single point.",
    },

NOTE: to be sure I get the right point, I like to set the start/end times 1 min before/after the points I am trying to flag. Because the data is in 10 min, this is just a precaution to make sure I catch the point(s) I am trying to flag.


In [ ]:
# Manual QC windows for `proc_2`.
# Add or edit these after reviewing `proc_1`.
manual_qc_flags = []


## run_proc_2

Applies manual_qc_flags to create the final IMOS FV01 product.


In [ ]:
workflow_config = require_ready_config()
if proc2_input is None:
    raise ValueError("Run the proc_1 save cell before proc_2.")

proc2_result = run_proc2(workflow_config)


## Review proc_2

Inspect the final QC-applied IMOS FV01 output before the compliance gate step.


In [ ]:
review_figure = plot_data_by_qc(
    proc2_result["dataset"],
    title=f"proc_2 FV01 review by QC flag: RBRQ {selected_metadata_row.get('inst_id')}",
)
review_figure.show()


## Save proc_2

Record the saved IMOS FV01 deliverable candidate. This file will be sent to the compliance gate.


In [ ]:
proc2_output_path = Path(proc2_result["output_path"])

# optional: make the parent directory shared/group-writable
proc2_output_path.parent.chmod(0o2775) #

# set the group ownership on the file itself
subprocess.run(["chgrp", "1054842", str(proc2_output_path)], check=True)

print(f"proc_2 FV01 saved: {proc2_output_path}")
print(f"QC log: {proc2_result['manual_qc_log']}")


# IMOS Delivery

This is the compliance gate for the final product.
Only the IMOS FV01 proc_2 product is staged as a deliverable and checked for IMOS compliance.


In [ ]:
workflow_config = require_ready_config()
delivery_result = run_imos_delivery(workflow_config)

# Set permissions on final deliverable
if delivery_result['proc_2_delivery'] is not None:
    delivery_path = Path(delivery_result['proc_2_delivery'])
    delivery_path.parent.chmod(0o2775)
    subprocess.run(["chgrp", "1054842", str(delivery_path)], check=True)

print(f"Delivery file: {delivery_result['imos_deliverables_file']}")
